In [1]:
import os
import tensorflow as tf
import numpy as np
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout, Flatten
from tensorflow.keras import Model
from keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

TYPE 1 FINE TUNNING

In [2]:
vgg=VGG16(
    weights='imagenet',
    include_top=True,
    input_shape=(224,224,3)
)

553467096/553467096 [==============================] - 657s 1us/step


In [ ]:
for layer in vgg.layers:
    print(f"{layer.name:<20} |TYPE:({type(layer).__name__}) |Trainable: {layer.trainable}")

In [ ]:
vgg.summary()

In [ ]:
model=Sequential()
model.add(vgg)
model.add(Dense(256,activation='sigmoid'))

In [ ]:
#generators
train_ds=keras.utils.image_dataset_from_directory(
    directory='dataset/train',
    labels='inferred',
    label_mode='int',
    image_size=(224,224),
    batch_size=32
)
validation_ds=keras.utils.image_dataset_from_directory(
    directory='dataset/validation',
    labels='inferred',
    label_mode='int',
    image_size=(224,224),
    batch_size=32
)

In [ ]:
#Normalize
def process(image,label):
    image=tf.cast(image/255.,tf.float32)
    return image,label

In [ ]:
train_ds=train_ds.map(process)
validation_ds=validation_ds.map(process)

In [ ]:
model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=0.0001),
    loss="BinaryCrossentropy",
    metrics=["accuracy"]
)

In [ ]:
history=model.fit(train_ds,validation_data=validation_ds,epochs=10,verbose=1)

In [ ]:
plt.plot(history.history['accuracy'],label='train_acc')
plt.plot(history.history['val_accuracy'],label='val_acc')
plt.legend()
plt.show()

In [ ]:
#testing
img_path=""
img=image.load_img(img_path,target_size=(224,224))
img_array=image.img_to_array(img)
img_array=np.expand_dims(img_array,axis=0)
img_array=img_array/255.

In [ ]:
prediction=model.predict(img_array)


In [ ]:
label="Dog" if prediction[0][0]>0.5 else "Cat"
confidence=prediction[0][0] 


In [ ]:
plt.imshow(img)
plt.title(f"{label} ({confidence:.2f})")
plt.axis('off')
plt.show()

TYPE 2 FINE-TUNNING

In [ ]:
vgg_conv_base=VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
    
)

In [ ]:
for layer in vgg_conv_base.layers:
    print(f"{layer.name:<20} |TYPE:({type(layer).__name__}) |Trainable: {layer.trainable}")

In [ ]:
vgg_conv_base.summary()

In [ ]:
model=Sequential()
model.add(vgg_conv_base)
model.add(Flatten())
model.add(Dense(256,activation='relu'))
model.add(Dense(1,activation='sigmoid'))


In [ ]:
#generaors
train_ds=keras.utils.image_dataset_from_directory(
    directory='dataset/train',
    labels='inferred',
    label_mode='int',
    image_size=(224,224),
    batch_size=32
)
validation_ds=keras.utils.image_dataset_from_directory(
    directory='dataset/validation',
    labels='inferred',
    label_mode='int',
    image_size=(224,224),
    batch_size=32
)

In [ ]:
def process(image,label):
    image=tf.cast(image/255.,tf.float32)
    return image,label

In [ ]:
train_ds=train_ds.map(process)
validation_ds=validation_ds.map(process)

In [ ]:
model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)